# Criação de origem_destino_customedio.csv

Esse notebook é utilizado para carregar os dados brutos e gerar a tabela `dados_processados/origem_destino_customedio.csv`. Essa tabela nos informa o preço médio cobrado, mensalmente, por cada compania aérea, para cada Origem -> Destino.

In [11]:
import glob
import pandas as pd 
import numpy as np  

A base de dados nos dá informações sobre quanto foi pago em tarifas aéreas no ano de 2026. Apesar de estar incompleta, tem dados até setembro, o que já serve bem. Esse código lê as tabelas sobre tarifas aéreas de cada mês disponível em 2026:

In [12]:
# sintaxe de generator function. funciona na base da fé!
df = pd.concat(
    (pd.read_csv(f, sep=';', encoding='latin-1') for f in glob.glob('../dados_brutos/*.csv')),
    ignore_index=True,
)
df.sample(5)
df.shape

(7414461, 7)

In [13]:
df.columns

df = df.rename(columns={'nr_ano_referencia': 'ano', 'nr_mes_referencia': 'mes', 'sg_empresa_icao': 'empresa',
                   'sg_icao_origem': 'origem', 'sg_icao_destino': 'destino', 'nr_tarifa': "valor_tarifa"})

Ajustando o tipo da coluna 'valor_tarifa' de str para float

In [14]:
display(df.dtypes)
# O valor da tarifa está como str!
print(df.columns)

def valor_tarifa_para_float(valor: str) -> float:
    valor = valor.replace(",", ".")
    valor = float(valor)
    valor = np.float16(valor)
    return valor

df['valor_tarifa'] = df['valor_tarifa'].apply(valor_tarifa_para_float)

df.dtypes

# Eba!!!

ano             int64
mes             int64
empresa           str
origem            str
destino           str
valor_tarifa      str
nr_assentos     int64
dtype: object

Index(['ano', 'mes', 'empresa', 'origem', 'destino', 'valor_tarifa',
       'nr_assentos'],
      dtype='str')


ano               int64
mes               int64
empresa             str
origem              str
destino             str
valor_tarifa    float16
nr_assentos       int64
dtype: object

Verificando dados faltantes (não há células em branco!)

In [15]:
print(f"Dados faltantes totais:\n{df.isna().sum()}")
# AEEE!!!

Dados faltantes totais:
ano             0
mes             0
empresa         0
origem          0
destino         0
valor_tarifa    0
nr_assentos     0
dtype: int64


In [16]:
df["custo_por_assento"] = df["valor_tarifa"] / df["nr_assentos"]
df.sample(5)
df.shape

(7414461, 8)

In [17]:
# reset_index faz ser retornado um df
origem_destino_customedio = df.groupby(["mes", "empresa", "origem", "destino"])["custo_por_assento"].mean().reset_index()
origem_destino_customedio

,mes,empresa,origem,destino,custo_por_assento
0,1,ABJ,SBSV,SDLO,346.126543
1,1,ABJ,SBSV,SNCL,190.978720
2,1,ABJ,SDLO,SBSV,321.578571
3,1,ABJ,SNCL,SBSV,269.143711
4,1,AZU,SBAE,SBAR,498.125000
...,...,...,...,...,...
96122,9,TAM,SBZM,SBSR,541.170455
96123,9,TAM,SBZM,SBSV,541.013141
96124,9,TAM,SBZM,SBTE,771.428571
96125,9,TAM,SBZM,SBUL,624.383333


In [18]:
origens_destinos_unicos = df.groupby(["origem", "destino"])["custo_por_assento"].mean().reset_index()
origens_destinos_unicos.to_csv(path_or_buf="../dados_processados/origens_destinos_unicos.csv", sep=";")

In [19]:
origens_destinos_unicos.sample(5)

,origem,destino,custo_por_assento
2911,SBHT,SBMO,1098.044025
1707,SBDB,SBBR,725.170743
6873,SBSV,SBPA,662.557775
922,SBCF,SBSV,543.871687
1708,SBDB,SBBV,1107.000000


Gravando o resultado na tabela `origem_destino_customedio.csv`.

In [ ]:
origem_destino_customedio.to_csv(path_or_buf="../dados_processados/origem_destino_customedio.csv", sep=";")

OSError: Cannot save file into a non-existent directory: 'dados_processados'

In [ ]:
origem_destino_customedio["empresa"].unique()